# Climate Data Cleaning: ISU Climate Data

Cleans the IEM/ISU daily climate summary (one row per station-day) into a tidy
table keyed on `station` + `day`, ready to spatially-and-temporally join onto
the EPA water-quality samples in the merge step.

**Input:**  `data/tabular/01_raw/climate/isu-climate.csv`
**Output:** `data/tabular/02_clean/climate/isu-climate-clean.csv`

**Pipeline**
1. Load the raw daily summary.
2. **Fix types & join key** — parse `day`, coerce numerics, drop rows with no
   `station`/`day` (they can never join), and enforce one row per
   `(station, day)` so the downstream left-join can't fan out.
3. **Range-validate** every measurement against physical bounds (RH 0–100,
   wind direction 0–360, plausible Iowa temperatures, non-negative precip), nulling
   impossible values instead of letting them reach the models.
4. **Cross-field consistency** — null pairs where `min > max` (e.g. a min temp
   above the day's max temp is a recording error).
5. **Missing values** — nothing is filled. Precipitation, snow and snow depth
   all keep their `NaN`s: their missingness is a property of *station
   instrumentation*, not of the weather (see below).
6. **Convert temperatures to Celsius** — every temperature-bearing column
   (`max_temp_f`, `min_temp_f`, `max_dewpoint_f`, `min_dewpoint_f`,
   `min_feel`/`avg_feel`/`max_feel`, `climo_high_f`, `climo_low_f`) arrives in
   Fahrenheit; convert to Celsius and rename with a `_c` suffix so units match
   the `prism_tmax_c`-style convention used by the PRISM cleaner.
7. Sanity-check and save.

**Why these changes matter** — the previous version (a) pointed at the old
`data/tabular/climate/{raw,clean}` paths that no longer exist, so it could not
run; (b) ran `dropna(subset=['max_temp_f','min_temp_f','precip_in'])`, which
discarded **117k of 221k rows (53%)** — almost entirely because `precip_in` was
missing. That missingness is *per-station*: roughly half the stations have no
precip sensor and report it ~20% of the time, while the other half report it
always. Dropping those rows threw away **113k rows that had perfectly good
temperature, humidity and wind data**; filling them with 0 would instead
fabricate "no rain" for half the network. Keeping the rows and leaving precip
`NaN` is the honest choice and roughly **doubles** climate coverage available to
the join. The old pipeline also did no range or consistency checking at all.

**Snow now gets the same treatment (Step 5).** The zero-fill this notebook used
to apply to `snow_in`/`snowd_in` was the very thing the precip argument above
rejects, and the snow columns have a *more* extreme version of the same
structure: **45 of 62 stations never report snow at all** and only eight carry
genuine snow data, while the null rate sits flat at ~86% in both January and
July. The fill manufactured ~190k zeros — about 87% of the column. Both columns
now keep their `NaN`s; genuinely reported zeros are still zeros.

> **Path note:** this targets the migrated `01_raw`/`02_clean` layout. The merge
> step (`src/03_merge/merge_epa_climate.py`) and README still read
> `data/tabular/climate/clean/isu-climate-clean.csv` and should be updated to
> point at `02_clean/climate/`. The `station` + `day` output contract is
> unchanged, so only the path needs updating there.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until we find the repo's data/tabular directory.

    Notebooks have no __file__, and the kernel's working directory varies
    (repo root vs. the notebook folder), so resolving paths relative to a
    fixed number of "../" is fragile — that fragility is exactly why the old
    notebook's hard-coded '../../../../data/...' paths broke after the layout
    migration. Searching upward for a sentinel makes the notebook runnable
    from anywhere.
    """
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "tabular").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing data/tabular/")


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "tabular" / "01_raw" / "climate"
CLEAN_DIR = REPO_ROOT / "data" / "tabular" / "02_clean" / "climate"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO_ROOT)
print("Raw dir:  ", RAW_DIR)
print("Clean dir:", CLEAN_DIR)

Repo root: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN/workspace/Water-Quality-Prediction
Raw dir:   /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN/workspace/Water-Quality-Prediction/data/tabular/01_raw/climate
Clean dir: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN/workspace/Water-Quality-Prediction/data/tabular/02_clean/climate


## Step 1 — Load

In [2]:
df = pd.read_csv(RAW_DIR / "isu-climate.csv")
n_raw = len(df)
print(f"Loaded {n_raw:,} station-day rows across {df['station'].nunique()} stations")
df.head()

Loaded 221,559 station-day rows across 62 stations


,station,day,max_temp_f,min_temp_f,max_dewpoint_f,min_dewpoint_f,precip_in,avg_wind_speed_kts,avg_wind_drct,min_rh,avg_rh,max_rh,snow_in,snowd_in,min_feel,avg_feel,max_feel,max_wind_speed_kts,climo_high_f,climo_low_f
0,OOA,2015-01-01,34.88,13.1,18.5,6.62,NaN,10.909408,244.36891,48.8897,69.056470,81.6625,NaN,NaN,-3.505770,10.060907,24.5662,18.0,32.9,15.6
1,ORC,2015-01-01,30.20,8.6,24.8,3.20,0.01,6.376307,253.90510,63.0255,75.334854,86.1759,NaN,NaN,-3.344350,13.669069,30.2000,15.0,27.2,10.5
2,AWG,2015-01-01,35.60,15.8,19.4,8.60,NaN,11.317073,242.09450,47.5111,67.679660,79.4681,NaN,NaN,-0.221851,12.198088,26.5835,17.0,31.0,13.8
3,CSQ,2015-01-01,33.80,10.4,19.4,5.00,NaN,9.979095,245.56772,54.5910,69.786600,85.0468,NaN,NaN,-4.830110,9.026784,24.7411,17.0,31.6,12.5
4,EBS,2015-01-01,32.00,10.4,23.0,5.00,NaN,10.198607,252.59528,63.7541,76.983210,92.6853,NaN,NaN,-7.222420,9.500152,22.9681,17.0,27.1,9.0


## Step 2 — Fix types & enforce the join key

Parse `day` to datetime and coerce every measurement column to numeric. Rows
with no `station` or no parseable `day` can never join onto a water-quality
sample, so they are dropped. Finally we **guarantee one row per
`(station, day)`** (averaging any exact duplicates): the merge step does a
left-join on this key, and duplicate keys would silently multiply
water-quality rows.

In [3]:
df["day"] = pd.to_datetime(df["day"], errors="coerce")

NUM_COLS = [
    "max_temp_f", "min_temp_f", "max_dewpoint_f", "min_dewpoint_f",
    "precip_in", "avg_wind_speed_kts", "avg_wind_drct",
    "min_rh", "avg_rh", "max_rh",
    "snow_in", "snowd_in", "min_feel", "avg_feel", "max_feel",
    "max_wind_speed_kts", "climo_high_f", "climo_low_f",
]
for col in NUM_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop rows that can never join (missing key), reporting the loss.
before = len(df)
df = df[df["station"].notna() & df["day"].notna()].copy()
print(f"Dropped rows with missing station/day: {before - len(df):,}  ({before:,} -> {len(df):,})")

# Enforce uniqueness of the join key.
n_dupes = df.duplicated(["station", "day"]).sum()
if n_dupes:
    print(f"Collapsing {n_dupes:,} duplicate (station, day) rows via mean")
    df = df.groupby(["station", "day"], as_index=False)[NUM_COLS].mean()
else:
    print("(station, day) is already unique — no collapsing needed")
assert not df.duplicated(["station", "day"]).any()

Dropped rows with missing station/day: 0  (221,559 -> 221,559)
(station, day) is already unique — no collapsing needed


## Step 3 — Range validation

All physical bounds live in one declarative dict. Values outside their bound are
**nulled** (not dropped — a single bad humidity reading shouldn't cost us the
day's temperature). Bounds are deliberately generous Iowa extremes; the goal is
to catch sensor garbage (e.g. a 600°F "feels-like"), not to trim the tails.

In [4]:
# column : (low, high) physically plausible bounds
RANGES = {
    "max_temp_f": (-60, 130), "min_temp_f": (-60, 130),
    "max_dewpoint_f": (-60, 100), "min_dewpoint_f": (-60, 100),
    "min_feel": (-80, 140), "avg_feel": (-80, 140), "max_feel": (-80, 140),
    "climo_high_f": (-60, 130), "climo_low_f": (-60, 130),
    "min_rh": (0, 100), "avg_rh": (0, 100), "max_rh": (0, 100),
    "avg_wind_drct": (0, 360),
    "avg_wind_speed_kts": (0, 150), "max_wind_speed_kts": (0, 250),
    "precip_in": (0, 30), "snow_in": (0, 60), "snowd_in": (0, 200),
}

total_nulled = 0
for col, (lo, hi) in RANGES.items():
    bad = df[col].notna() & ~df[col].between(lo, hi)
    if bad.sum():
        print(f"{col:<20} nulled {int(bad.sum()):>4} value(s) outside [{lo}, {hi}]")
    df.loc[bad, col] = np.nan
    total_nulled += int(bad.sum())
print(f"\nTotal out-of-range values nulled: {total_nulled:,}")

max_temp_f           nulled   21 value(s) outside [-60, 130]
min_temp_f           nulled    6 value(s) outside [-60, 130]
max_dewpoint_f       nulled    5 value(s) outside [-60, 100]
min_dewpoint_f       nulled   15 value(s) outside [-60, 100]
min_feel             nulled    5 value(s) outside [-80, 140]
avg_feel             nulled    7 value(s) outside [-80, 140]
max_feel             nulled   53 value(s) outside [-80, 140]
max_wind_speed_kts   nulled    4 value(s) outside [0, 250]

Total out-of-range values nulled: 116


## Step 4 — Cross-field consistency

A day's recorded minimum cannot exceed its maximum. Where it does, both ends are
recording errors, so we null the pair rather than trust either.

In [5]:
MIN_MAX_PAIRS = [
    ("min_temp_f", "max_temp_f"),
    ("min_dewpoint_f", "max_dewpoint_f"),
    ("min_rh", "max_rh"),
    ("min_feel", "max_feel"),
]
for lo_col, hi_col in MIN_MAX_PAIRS:
    bad = df[lo_col].notna() & df[hi_col].notna() & (df[lo_col] > df[hi_col])
    if bad.sum():
        print(f"{lo_col} > {hi_col}: nulled {int(bad.sum())} inconsistent pair(s)")
    df.loc[bad, [lo_col, hi_col]] = np.nan

min_temp_f > max_temp_f: nulled 1 inconsistent pair(s)
min_rh > max_rh: nulled 2 inconsistent pair(s)


## Step 5 — Missing values

**Nothing is filled here.** Every column keeps its `NaN`s; per-column gaps are
handled at modeling time, and a gap in one field shouldn't discard the whole row.

- **Precipitation is left `NaN`.** Its 53% missingness is driven by *station
  instrumentation*, not by dry days — about half the stations report precip only
  ~20% of the time (no precip sensor) while the rest report it nearly always.
  Filling 0 would fabricate "no rain" for those stations and bias any
  precip-dependent model; dropping the rows (the old behaviour) would throw away
  their temperature/humidity/wind data. Leaving `NaN` lets the modeling step
  impute or branch on availability.

- **Snow & snow depth are left `NaN` too — this is a fix.** Earlier versions of
  this notebook filled them with 0, on the theory that they are *event*
  variables the feed reports only when snow occurs, so a blank means "no snow".
  **The data does not support that.** Three checks, all in the diagnostic below:

  1. **The feed already has a representation for zero.** `snow_in` carries
     28,434 explicit `0.0` values. A blank is therefore *not* how "no snow" is
     encoded — zero is.
  2. **Missingness has no seasonal structure.** The `snow_in` null rate is
     **86.5% in January and 85.9% in July**, spanning under 3 percentage points
     across the whole year. A genuine event variable would span ~50pp.
     Missingness is not tracking the weather.
  3. **It is a station property.** Of 62 stations, **45 never report `snow_in`
     at all** in ten years, one reports only `0.0`, and eight more report only a
     `0.0001`-inch sentinel — never real snow. **Just eight stations carry
     genuine snow data**, and only six of those (DSM, DVN, DBQ, ALO, SUX, MCW —
     the major staffed airports) cover ~84% of days. Their data behaves exactly
     as Iowa should: mean 0.42 in/day in January, exactly 0.0 in June and August.

  So the fill was asserting that 45 stations with no snow instrument measured
  zero snowfall every day for a decade, blizzards included. It fabricated
  **189,974** values against **28,434** genuinely reported zeros — about 87% of
  the resulting column. That is the same failure the precip bullet above already
  guards against; snow simply has a more extreme version of the identical
  station-instrumentation structure.

  Genuine reported zeros are *kept* as zeros — they are real measurements. Only
  the blanks stay blank.

> Downstream, `isu_snow_in` / `isu_snowd_in` become mostly-`NaN` rather than
> mostly-`0`. Both tree families impute internally
> (`HistGradientBoostingRegressor` handles `NaN` natively), so this is safe —
> and `eda-summary.md` §4.1 independently recommends dropping both columns from
> `FEATURE_COLS` on the grounds that they carry no signal. That recommendation
> was made against the *fabricated* column; the honest one is what should be
> re-evaluated.

In [6]:
# No fills. The diagnostic below is the evidence for that choice on the two
# columns most likely to be "helpfully" zero-filled again later: a blank in
# snow_in/snowd_in means the station never measured snow, not that no snow fell.
EVENT_LIKE = ["snow_in", "snowd_in", "precip_in"]
TRACE_IN = 0.001  # stations whose max snowfall is a 1e-4 sentinel, never real snow

print("Why snow/precip blanks are NOT zeros")
print("=" * 62)

# 1. Zero has its own representation, so a blank cannot also mean zero.
print("\n1. The feed encodes zero explicitly:")
for col in EVENT_LIKE:
    print(f"   {col:<10} NaN {df[col].isna().sum():>7,}  "
          f"==0 {int((df[col] == 0).sum()):>6,}  "
          f">0 {int((df[col] > 0).sum()):>6,}")

# 2. If a blank meant "no event", missingness would be strongly seasonal.
print("\n2. Null rate by month (flat => missingness is not the weather):")
by_month = df.assign(month=df["day"].dt.month).groupby("month")
null_rate = by_month[EVENT_LIKE].apply(lambda g: g.isna().mean())
print("   mon " + "".join(f"{c:>12}" for c in EVENT_LIKE))
for m in range(1, 13):
    print(f"   {m:>3} " + "".join(f"{null_rate.loc[m, c]:>11.1%}" for c in EVENT_LIKE))
snow_spread = null_rate["snow_in"].max() - null_rate["snow_in"].min()
print(f"   snow_in null-rate spread across the year: {snow_spread:.1%} "
      f"(a genuine event variable would span ~50pp)")

# 3. Missingness is a station property: most stations have no snow instrument.
print("\n3. Per-station snow_in reporting:")
by_station = df.groupby("station")["snow_in"]
reporting = by_station.apply(lambda s: s.notna().mean())
peak = by_station.max()
n_stations = len(reporting)

never = reporting == 0
zero_only = (reporting > 0) & (peak.fillna(0) == 0)
trace_only = (peak > 0) & (peak <= TRACE_IN)
real = peak > TRACE_IN

print(f"   never report snow_in at all:      {int(never.sum()):>3} of {n_stations}")
print(f"   report, but only ever 0.0:        {int(zero_only.sum()):>3} of {n_stations}")
print(f"   report, but only a <=1e-3 trace:  {int(trace_only.sum()):>3} of {n_stations}"
      f"  ({', '.join(sorted(peak[trace_only].index))})")
print(f"   carry real snow data:             {int(real.sum()):>3} of {n_stations}")
for st in reporting[real].sort_values(ascending=False).index:
    print(f"      {st}  reports {reporting[st]:>5.1%} of days | peak {peak[st]:>5.1f} in")
print(f"   ...of which report >=80% of days: {int((reporting >= 0.8).sum()):>3}")

assert df["snow_in"].isna().any(), "snow_in was filled — see Step 5, do not zero-fill"
assert df["snowd_in"].isna().any(), "snowd_in was filled — see Step 5, do not zero-fill"

miss = df[NUM_COLS].isna().sum()
miss_pct = (miss / len(df) * 100).round(1)
print("\nMissing values per column (count, %):")
print(pd.concat([miss.rename("missing"), miss_pct.rename("pct")], axis=1).to_string())

Why snow/precip blanks are NOT zeros

1. The feed encodes zero explicitly:
   snow_in    NaN 189,974  ==0 28,434  >0  3,151
   snowd_in   NaN 199,351  ==0 19,500  >0  2,708
   precip_in  NaN 117,564  ==0 43,055  >0 60,940

2. Null rate by month (flat => missingness is not the weather):
   mon      snow_in    snowd_in   precip_in
     1       86.5%      92.6%      59.3%
     2       86.3%      93.0%      61.1%
     3       86.6%      92.0%      53.0%
     4       86.4%      89.9%      49.4%
     5       85.7%      88.8%      42.9%
     6       85.9%      89.4%      48.3%
     7       85.9%      89.4%      51.9%
     8       85.9%      89.4%      51.8%
     9       86.0%      89.4%      52.3%
    10       85.8%      89.0%      53.1%
    11       84.2%      88.5%      55.9%
    12       83.7%      88.5%      58.3%
   snow_in null-rate spread across the year: 2.9% (a genuine event variable would span ~50pp)

3. Per-station snow_in reporting:
   never report snow_in at all:       45 of 62
 

## Step 6 — Convert temperatures to Celsius

Every temperature-bearing column in the raw feed — `max_temp_f`, `min_temp_f`,
`max_dewpoint_f`, `min_dewpoint_f`, `climo_high_f`, `climo_low_f`, and the
unsuffixed `min_feel`/`avg_feel`/`max_feel` "feels-like" fields — is reported in
Fahrenheit. Convert them to Celsius and rename with a `_c` suffix so units are
self-documenting and consistent with the PRISM cleaner's `prism_tmax_c` /
`prism_tmin_c` convention: nothing downstream should have to guess a column's
unit from its name, or silently mix °F and °C across climate sources in the
merge step.

In [7]:
# Fahrenheit-native columns -> Celsius, renamed with the `_c` unit suffix used
# elsewhere in the pipeline (e.g. PRISM's `prism_tmax_c`).
FAHRENHEIT_TO_CELSIUS = {
    "max_temp_f": "max_temp_c",
    "min_temp_f": "min_temp_c",
    "max_dewpoint_f": "max_dewpoint_c",
    "min_dewpoint_f": "min_dewpoint_c",
    "min_feel": "min_feel_c",
    "avg_feel": "avg_feel_c",
    "max_feel": "max_feel_c",
    "climo_high_f": "climo_high_c",
    "climo_low_f": "climo_low_c",
}

for f_col in FAHRENHEIT_TO_CELSIUS:
    df[f_col] = (df[f_col] - 32) * 5 / 9

df = df.rename(columns=FAHRENHEIT_TO_CELSIUS)

TEMP_C_COLS = list(FAHRENHEIT_TO_CELSIUS.values())
print("Converted to Celsius and renamed:", ", ".join(TEMP_C_COLS))
df[TEMP_C_COLS].describe().round(2)

Converted to Celsius and renamed: max_temp_c, min_temp_c, max_dewpoint_c, min_dewpoint_c, min_feel_c, avg_feel_c, max_feel_c, climo_high_c, climo_low_c


,max_temp_c,min_temp_c,max_dewpoint_c,min_dewpoint_c,min_feel_c,avg_feel_c,max_feel_c,climo_high_c,climo_low_c
count,217200.00,217251.00,217052.00,217055.00,216858.00,218477.00,216808.00,221559.00,221559.00
mean,15.75,4.53,8.39,1.65,1.79,8.08,15.01,15.12,3.46
std,12.31,11.29,10.85,11.77,14.06,14.16,14.47,11.07,10.28
min,-36.39,-38.22,-47.00,-51.00,-58.66,-46.78,-46.33,-5.44,-15.33
25%,6.00,-3.00,0.00,-6.10,-7.86,-2.78,3.01,4.39,-6.00
50%,17.78,5.00,8.00,1.00,2.13,9.93,17.50,16.67,3.61
75%,26.67,14.44,18.00,12.00,14.61,20.37,26.59,26.06,13.61
max,54.00,37.00,37.00,37.00,57.92,47.81,59.66,31.33,20.22


## Step 7 — Sanity check

Confirm the headline fields now sit in sensible ranges and that we retained the
rows the old pipeline was discarding.

In [8]:
print(f"Rows retained: {len(df):,} of {n_raw:,} raw "
      f"({len(df) / n_raw:.0%}) — old pipeline kept 103,803 ({103803 / n_raw:.0%})")
print(f"Date range: {df['day'].min().date()} → {df['day'].max().date()}\n")
df[["max_temp_c", "min_temp_c", "precip_in", "avg_rh", "avg_wind_drct"]].describe().round(2)

Rows retained: 221,559 of 221,559 raw (100%) — old pipeline kept 103,803 (47%)
Date range: 2015-01-01 → 2024-12-31



,max_temp_c,min_temp_c,precip_in,avg_rh,avg_wind_drct
count,217200.00,217251.00,103995.00,216853.00,218484.00
mean,15.75,4.53,0.15,74.39,196.08
std,12.31,11.29,0.39,13.89,99.65
min,-36.39,-38.22,0.00,1.00,0.00
25%,6.00,-3.00,0.00,65.51,127.70
50%,17.78,5.00,0.00,75.41,190.89
75%,26.67,14.44,0.12,84.47,289.86
max,54.00,37.00,30.00,100.00,360.00


## Step 8 — Save

In [9]:
out_file = CLEAN_DIR / "isu-climate-clean.csv"
df.to_csv(out_file, index=False)
print(f"Saved {len(df):,} station-day rows -> {out_file}")

Saved 221,559 station-day rows -> /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN/workspace/Water-Quality-Prediction/data/tabular/02_clean/climate/isu-climate-clean.csv
